# 03 · Requirement tags review (step 5)

Checks the requirement dictionary (`requirement_dict`: one row per distinct phrase and column) and the tags derived from it
(`ad_requirements`: one row per phrase mention and tag). Needs steps 1–5; the last section also uses the positions from step 4.

Counts use **distinct ads** (canonical printing, step 3) without death register entries (step 2).

In [ ]:
import pandas as pd
from IPython.display import display
from hisrag.data import query

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.max_rows", 60)

BASE = """
    ad_requirements r
    JOIN ads a USING (ad_id)
    JOIN ad_dups d USING (ad_id)
    JOIN ad_text t USING (ad_id)
    WHERE d.is_canonical AND NOT t.flag_death_register
"""

query("""SELECT "column", count(*) AS phrases, sum("count") AS mentions,
                  round(100 * avg(is_informative::int), 1) AS pct_with_tags
           FROM requirement_dict GROUP BY 1 ORDER BY mentions DESC""")

## Tags per dimension (distinct ads)

In [ ]:
query(f"""SELECT r."group", r.dimension, count(DISTINCT r.ad_id) AS ads, count(DISTINCT r.value) AS distinct_values
            FROM {BASE} GROUP BY ALL ORDER BY r."group", ads DESC""")

## Most frequent values of a dimension

In [ ]:
def top_values(dimension: str, n: int = 20):
    return query(f"""SELECT r.value, count(DISTINCT r.ad_id) AS ads
                       FROM {BASE} AND r.dimension = ? GROUP BY 1 ORDER BY ads DESC LIMIT {n}""", [dimension])
top_values("fachkenntnisse")

In [ ]:
top_values("bewerbungsformalitaeten")

In [ ]:
top_values("alter")

## Details dropped by the grounding check
The check keeps a detail only if its words occur in the phrase, allowing historical spellings (correcte → korrekt, nähe → nahe)
and a few cue words (möglichst → bevorzugt, absolvirt → abgeschlossen). The number dropped is in the run report
(`details_dropped_as_ungrounded`; 2,797 before the spelling rules). Rows here show the model's detail (`detail_raw`) next to the phrase.

In [ ]:
def dropped_details(n: int = 25, seed: int = 0):
    return query(f"""SELECT d."column", d.surface, d."count", tg.dimension, tg.value, tg.detail_raw
                       FROM requirement_dict d, UNNEST(d.tags) AS u(tg)
                       WHERE tg.detail_raw IS NOT NULL AND tg.detail IS NULL
                       ORDER BY hash(d.key || '{seed}') LIMIT {n}""")
dropped_details()

In [ ]:
# Most frequent dropped details (weighted by mentions)
query("""SELECT tg.detail_raw, sum(d."count") AS mentions, count(*) AS phrases
           FROM requirement_dict d, UNNEST(d.tags) AS u(tg)
           WHERE tg.detail_raw IS NOT NULL AND tg.detail IS NULL
           GROUP BY 1 ORDER BY mentions DESC LIMIT 20""")

## Values that do not occur in the phrase
In the audit of dropped details, about a quarter of the sampled tags took their **value** from the context, not only the detail
(`"ger"` → Buchführung, `"Deutsch und Französisch"` → Latein). This flag marks tags none of whose value words occur in the
phrase. It also catches legitimate modern paraphrases (`"hiesigen Platz"` → Ortskenntnis), so read the sample before deciding
whether to drop such tags, and in which cases (e.g. only short fragments, or only when the detail was dropped as well).

In [ ]:
from hisrag.normalize.requirements import value_is_grounded

tags = query("""SELECT d.key, d."column", d.surface, d."count", length(d.phrase_key) AS key_len,
                         tg.dimension, tg.value, tg.detail, tg.detail_raw
                  FROM requirement_dict d, UNNEST(d.tags) AS u(tg)""")
tags["value_grounded"] = [value_is_grounded(v, s) for v, s in zip(tags["value"], tags["surface"])]
tags["detail_dropped"] = tags["detail_raw"].notna() & tags["detail"].isna()
tags["short_phrase"] = tags["key_len"] < 6

# Share of tag mentions whose value is not in the phrase, split by the two cheap signals
summary = (tags.assign(mentions=tags["count"])
               .groupby(["short_phrase", "detail_dropped"])
               .apply(lambda g: pd.Series({"tags": len(g), "mentions": g.mentions.sum(),
                                           "pct_value_ungrounded": round(100 * (~g.value_grounded * g.mentions).sum() / g.mentions.sum(), 1)}),
                      include_groups=False))
summary

In [ ]:
# Sample of ungrounded values, most frequent phrases first: which are context leaks, which are fair paraphrases?
tags.loc[~tags.value_grounded, ["column", "surface", "count", "dimension", "value", "detail_raw"]]     .sort_values("count", ascending=False).head(40)

In [ ]:
# The same for a random sample (rare phrases)
tags.loc[~tags.value_grounded, ["column", "surface", "count", "dimension", "value", "detail_raw"]].sample(25, random_state=0)

## Values outside the closed lists

In [ ]:
query("""SELECT tg.dimension, tg.value, sum(d."count") AS mentions, any_value(d.surface) AS example
           FROM requirement_dict d, UNNEST(d.tags) AS u(tg)
           WHERE NOT tg.in_vocab GROUP BY 1, 2 ORDER BY mentions DESC LIMIT 25""")

## First analysis: personal requirements by gender of the position title
Share of ads (job offers vs. job searches) whose position wording is female or male (`gender_form` from step 4) that mention
a given Person/Eigenschaften dimension. Ads naming positions of both genders are left out. Read this as a first look, not as a
result: tags come from automatic extraction, and small cells are unreliable (the `ads` column gives the base).

In [ ]:
g = query(f"""
    WITH pos AS (
        SELECT ad_id, CASE WHEN bool_and(gender_form = 'f') THEN 'f'
                           WHEN bool_and(gender_form = 'm') THEN 'm' END AS gender
        FROM ad_positions WHERE gender_form IN ('f', 'm') GROUP BY ad_id
    ),
    base AS (
        SELECT a.ad_id, a.label, pos.gender
        FROM ads a JOIN pos USING (ad_id) JOIN ad_dups d USING (ad_id) JOIN ad_text t USING (ad_id)
        WHERE d.is_canonical AND NOT t.flag_death_register AND pos.gender IS NOT NULL
          AND a.label IN ('job_offer', 'job_search')
    ),
    tagged AS (SELECT DISTINCT ad_id, dimension FROM ad_requirements WHERE "group" IN ('Person', 'Eigenschaften'))
    SELECT b.label, b.gender, tg.dimension,
           count(DISTINCT tg.ad_id) AS ads_with_tag,
           (SELECT count(*) FROM base b2 WHERE b2.label = b.label AND b2.gender = b.gender) AS ads
    FROM base b JOIN tagged tg USING (ad_id)
    GROUP BY ALL""")
g["pct"] = (100 * g.ads_with_tag / g.ads).round(1)
g.pivot_table(index="dimension", columns=["label", "gender"], values="pct").round(1)

In [ ]:
# The bases: how many ads per label and gender
g.drop_duplicates(["label", "gender"])[["label", "gender", "ads"]]